# 06 — Ensemble learning | HAI 21.03

**Pergunta do experimento:** combinar a Regressão Logística com o Isolation Forest com PCA sinaliza registros de ataque que a Logística sozinha deixa passar? Quantos falsos alarmes a combinação acrescenta?

Cada modelo já treinado vê a mesma linha de medições e dá sua resposta: `0` (sem alerta) ou `1` (alerta). Vamos juntar essas duas respostas pela regra **OU**. É uma combinação de decisões binárias, também chamada de *hard voting*. Com dois modelos, quando só um alerta, a nossa regra dá prioridade ao alerta.

**Para rodar no VS Code:** salve este arquivo em `tcc-hai_modular/notebooks/`, escolha a mesma `.venv` e execute as células de cima para baixo. Você precisa ter as previsões dos notebooks 03 e 04 da versão modular e as da execução de `executar_regressao_logistica.py`. As tabelas que já vêm no notebook são de uma execução de referência. Ao rodar, você verá os resultados dos seus arquivos.

### 1. Localizar as duas execuções anteriores

As previsões do IF com PCA vêm da execução dos notebooks 03 e 04. As da Logística vêm do script da etapa 4. A célula procura as duas dentro de `runs/`. Se encontrar mais de uma opção, coloque o nome da pasta que você quer usar nas variáveis abaixo, sem escrever `runs/` na frente.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import sys

import pandas as pd
from IPython.display import display

EXECUCAO_IF = None         # Ex.: "baseline_modular"
EXECUCAO_LOGISTICA = None  # Ex.: "regressao_logistica_20261111_123657"

caminhos = [Path.cwd(), *Path.cwd().parents]
raiz = next((p for p in caminhos if (p / "src/hai_tcc/evaluation.py").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Abra a pasta tcc-hai_modular e coloque o notebook em notebooks/.")
if str(raiz / "src") not in sys.path:
    sys.path.insert(0, str(raiz / "src"))
from hai_tcc.evaluation import aplicar_limiar, calcular_metricas, identificar_intervalos

runs = raiz / "runs"
VALIDACAO_CSV = "validacao_previsoes.csv.gz"
PRED_LR = "previsao_regressao_logistica"
PRED_IF = "previsao_if_com_pca"
PRED_OU = "previsao_ensemble_ou"
SCORE_LR = "pontuacao_regressao_logistica"
SCORE_IF = "pontuacao_if_com_pca"
LIMIAR_IF = "limiar_if_com_pca"

def localizar_execucao(nome, variavel, colunas, instrucao):
    arquivos = (runs.glob("*/previsoes_e_metricas/" + VALIDACAO_CSV) if nome is None
                else [runs / nome / "previsoes_e_metricas" / VALIDACAO_CSV])
    candidatas = [p.parent for p in arquivos if p.is_file()
                  and colunas.issubset(pd.read_csv(p, nrows=0).columns)]
    if not candidatas:
        raise FileNotFoundError(f"Não encontrei as previsões para {variavel}. {instrucao}")
    if len(candidatas) > 1:
        nomes = ", ".join(sorted(p.parent.name for p in candidatas))
        raise ValueError(f"Preencha {variavel} com uma destas pastas de runs/: {nomes}.")
    return candidatas[0]

pasta_if = localizar_execucao(
    EXECUCAO_IF, "EXECUCAO_IF", {PRED_IF, SCORE_IF, LIMIAR_IF},
    "Confira a pasta da versão modular ou execute os notebooks 03 e 04.",
)
pasta_lr = localizar_execucao(
    EXECUCAO_LOGISTICA, "EXECUCAO_LOGISTICA", {PRED_LR, SCORE_LR},
    "Confira a pasta da etapa 4 ou execute executar_regressao_logistica.py.",
)
print("IF com PCA:", pasta_if.relative_to(raiz))
print("Regressão Logística:", pasta_lr.relative_to(raiz))

IF com PCA: runs/baseline_modular/previsoes_e_metricas
Regressão Logística: runs/regressao_logistica_20261111_123657/previsoes_e_metricas


### 2. Juntar as respostas para a mesma medição

Imagine duas listas de respostas para a mesma prova. Antes de comparar, precisamos garantir que a resposta de uma lista corresponde à mesma questão da outra. Aqui, a chave é **arquivo de origem + índice da linha + horário**.

A célula confere se as duas listas trazem todas as linhas, se os rótulos reais coincidem e se os alertas foram produzidos com os limiares salvos. `attack_real` é o gabarito usado para medir os acertos.

In [2]:
CHAVE = ["arquivo_origem", "indice_origem", "time"]
REAL = "attack_real"
ARQUIVOS = {
    "test4": (VALIDACAO_CSV, VALIDACAO_CSV, 39601, 1157),
    "test5": ("teste_historico_previsoes.csv.gz", "teste_previsoes.csv.gz", 92401, 2177),
}

def carregar_particao(particao):
    arq_lr, arq_if, total, ataques = ARQUIVOS[particao]

    def ler(pasta, nome, colunas):
        caminho = pasta / nome
        if not caminho.is_file():
            raise FileNotFoundError(f"Faltou {caminho.relative_to(raiz)}. Confira a execução escolhida.")
        return pd.read_csv(caminho, usecols=CHAVE + [REAL] + colunas,
                           parse_dates=["time"], float_precision="round_trip")

    lr = ler(pasta_lr, arq_lr, [PRED_LR, SCORE_LR])
    ifp = ler(pasta_if, arq_if, [PRED_IF, SCORE_IF, LIMIAR_IF])
    dados = lr.merge(ifp, on=CHAVE, how="outer", validate="one_to_one",
                     suffixes=("", "_if"), indicator=True)
    if (len(dados) != total or not dados["_merge"].eq("both").all()
            or not dados["arquivo_origem"].eq(f"{particao}.csv.gz").all()
            or dados[CHAVE].isna().any().any()):
        raise ValueError(f"Os arquivos não correspondem às {total} linhas esperadas de {particao}.")
    if (not dados[REAL].eq(dados[f"{REAL}_if"]).all()
            or not dados[[REAL, PRED_LR, PRED_IF]].isin([0, 1]).all().all()
            or dados[REAL].sum() != ataques):
        raise ValueError(f"Rótulos ou previsões incompatíveis em {particao}.")
    return dados.drop(columns=[f"{REAL}_if", "_merge"]).sort_values("time").reset_index(drop=True)

validacao = carregar_particao("test4")
selecao_lr = json.loads((pasta_lr / "selecao_validacao.json").read_text(encoding="utf-8"))
if selecao_lr["arquivo"] != "test4.csv.gz":
    raise ValueError("A Logística precisa ter o limiar definido no test4.")
limiar_lr = float(selecao_lr["limiar"])
if validacao[LIMIAR_IF].nunique(dropna=False) != 1:
    raise ValueError("O IF com PCA precisa ter um único limiar no test4.")
limiar_if = float(validacao[LIMIAR_IF].iloc[0])

def conferir_limiares(dados):
    if (not dados[LIMIAR_IF].eq(limiar_if).all()
            or not (dados[PRED_IF] == aplicar_limiar(dados[SCORE_IF], limiar_if)).all()
            or not (dados[PRED_LR] == aplicar_limiar(dados[SCORE_LR], limiar_lr)).all()):
        raise ValueError("Os alertas não correspondem aos limiares salvos na validação.")

conferir_limiares(validacao)
print(f"test4: {len(validacao):,} linhas conferidas, {validacao[REAL].sum():,} de ataque.".replace(",", "."))
print(f"Limiares usados: Logística = {limiar_lr:.8f}; IF com PCA = {limiar_if:.8f}.")

test4: 39.601 linhas conferidas. 1.157 de ataque.
Limiares usados: Logística = 0.15296581; IF com PCA = 0.56799924.


### 3. Combinar com a regra OU e medir o resultado

| Logística | IF com PCA | Ensemble OU |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 1 |

**Macete:** o IF só acrescenta um alerta quando a Logística diz `0` e ele diz `1`. Aí olhamos o rótulo real: se era ataque, ganhamos uma detecção; se era normal, ganhamos um falso alarme. A regra OU conserva todos os alertas da Logística, inclusive os falsos.

O **recall** mostra a fração de linhas de ataque sinalizadas. A **precisão** mostra a fração de alertas que realmente eram ataque. O **F1** combina as duas medidas. Um **episódio** conta como detectado se houver pelo menos um alerta no trecho contínuo rotulado como ataque.

In [3]:
MODELOS = {
    "Regressão Logística": PRED_LR,
    "IF com PCA": PRED_IF,
    "Ensemble OU": PRED_OU,
}

def combinar_e_avaliar(dados):
    # A regra lê apenas os dois alertas, ambos já calculados pelos modelos.
    alerta = (dados[PRED_LR].eq(1) | dados[PRED_IF].eq(1)).astype(int)
    por_linha, intervalos = identificar_intervalos(dados.assign(**{PRED_OU: alerta}))
    linhas_ataque = por_linha.loc[por_linha[REAL].eq(1)]
    metricas, episodios = [], []
    for modelo, coluna in MODELOS.items():
        m = calcular_metricas(por_linha[REAL], por_linha[coluna])
        detectados = linhas_ataque.loc[linhas_ataque[coluna].eq(1)].groupby("intervalo_id")
        resumo = detectados["time"].agg(registros_detectados="size", primeiro_alerta="min")
        ep = intervalos.merge(resumo, on="intervalo_id", how="left")
        ep["registros_detectados"] = ep["registros_detectados"].fillna(0).astype(int)
        ep["atraso_segundos"] = (ep["primeiro_alerta"] - ep["inicio"]).dt.total_seconds()
        ep["cobertura"] = ep["registros_detectados"] / ep["registros_ataque"]
        episodios.append(ep.assign(modelo=modelo))
        atrasos = ep["atraso_segundos"].dropna()
        metricas.append({
            "Modelo": modelo, "VP": m["Verdadeiros positivos"], "FP": m["Falsos positivos"],
            "FN": m["Falsos negativos"], "Precisão": m["Precisão"], "Recall": m["Recall"],
            "F1": m["F1-score"], "Episódios detectados": len(atrasos), "Episódios totais": len(ep),
            "Atraso mediano (s)": atrasos.median() if len(atrasos) else float("nan"),
        })
    return por_linha, pd.DataFrame(metricas).set_index("Modelo"), pd.concat(episodios, ignore_index=True)

def mostrar_comparacao(por_linha, metricas):
    tabela = metricas[["VP", "FP", "FN", "Precisão", "Recall", "F1", "Atraso mediano (s)"]].copy()
    tabela["Episódios"] = metricas["Episódios detectados"].astype(str) + "/" + metricas["Episódios totais"].astype(str)
    display(tabela.round(4))
    novos_alertas = por_linha[PRED_LR].eq(0) & por_linha[PRED_IF].eq(1)
    recuperados = int((novos_alertas & por_linha[REAL].eq(1)).sum())
    falsos_extras = int((novos_alertas & por_linha[REAL].eq(0)).sum())
    delta_f1 = metricas.loc["Ensemble OU", "F1"] - metricas.loc["Regressão Logística", "F1"]
    print(f"Em relação à Logística: {recuperados} registros de ataque recuperados; "
          f"{falsos_extras} falsos alarmes extras; variação no F1 = {delta_f1:+.4f}.")

### 4. Conferir a combinação na validação (`test4`)

A tabela compara os dois modelos sozinhos e a regra OU. `VP` são linhas de ataque sinalizadas, `FP` são falsos alarmes e `FN` são linhas de ataque que passaram sem alerta. Nas duas linhas de exemplo abaixo, a Logística deu `0` e o IF deu `1`. Veja como o rótulo real muda a interpretação desse mesmo alerta.

In [4]:
linhas_val, metricas_val, episodios_val = combinar_e_avaliar(validacao)
mostrar_comparacao(linhas_val, metricas_val)

exemplos = linhas_val.loc[linhas_val[PRED_LR].eq(0) & linhas_val[PRED_IF].eq(1)]
exemplos = exemplos.groupby(REAL, sort=False).head(1)[["time", PRED_LR, PRED_IF, PRED_OU, REAL]]
exemplos = exemplos.rename(columns={"time": "Horário", PRED_LR: "Logística", PRED_IF: "IF com PCA",
                                    PRED_OU: "Ensemble OU", REAL: "Rótulo real"})
exemplos["Interpretação"] = exemplos["Rótulo real"].map({0: "Falso alarme extra", 1: "Ataque recuperado"})
display(exemplos.reset_index(drop=True))

,VP,FP,FN,Precisão,Recall,F1,Atraso mediano (s),Episódios
Modelo,,,,,,,,
Regressão Logística,781,70,376,0.9177,0.6750,0.7779,1.0,5/5
IF com PCA,622,156,535,0.7995,0.5376,0.6429,25.5,4/5
Ensemble OU,828,175,329,0.8255,0.7156,0.7667,1.0,5/5


Em relação à Logística: 47 registros de ataque recuperados; 105 falsos alarmes extras; variação no F1 = -0.0112.


,Horário,Logística,IF com PCA,Ensemble OU,Rótulo real,Interpretação
0,2020-07-28 12:04:13,0,1,1,0,Falso alarme extra
1,2020-07-28 12:43:26,0,1,1,1,Ataque recuperado


### 5. Aplicar a mesma regra ao teste histórico (`test5`)

A regra OU e os dois limiares continuam iguais. Como o `test5` já foi analisado nas etapas anteriores, esta é uma comparação histórica. Ela mostra como a combinação se comporta nesse período. Se quiser experimentar outro limiar ou outra regra, faça a escolha no `test4`.

A segunda tabela mostra, em cada episódio, quantas linhas de ataque foram sinalizadas e em quanto tempo saiu o primeiro alerta. Um `—` indica que não houve alerta dentro do trecho.

In [5]:
teste = carregar_particao("test5")
conferir_limiares(teste)
linhas_teste, metricas_teste, episodios_teste = combinar_e_avaliar(teste)
mostrar_comparacao(linhas_teste, metricas_teste)

detalhe = episodios_teste.pivot(index="intervalo_id", columns="modelo",
                                values=["registros_detectados", "atraso_segundos"])
detalhe = detalhe.rename_axis(index="Episódio").rename(columns={
    "registros_detectados": "Linhas sinalizadas", "atraso_segundos": "Primeiro alerta (s)",
}, level=0)
display(detalhe.astype("Int64").astype(object).where(detalhe.notna(), "—"))

,VP,FP,FN,Precisão,Recall,F1,Atraso mediano (s),Episódios
Modelo,,,,,,,,
Regressão Logística,1116,643,1061,0.6345,0.5126,0.5671,21.0,12/12
IF com PCA,900,827,1277,0.5211,0.4134,0.4611,24.0,10/12
Ensemble OU,1260,1118,917,0.5299,0.5788,0.5532,13.5,12/12


Em relação à Logística: 144 registros de ataque recuperados; 475 falsos alarmes extras; variação no F1 = -0.0138.


Linhas sinalizadas                                 \
modelo          Ensemble OU IF com PCA Regressão Logística   
Episódio                                                     
1                        86         63                  85   
2                        24         13                  15   
3                       154        153                 114   
4                       113         55                 113   
5                       131        115                 108   
6                       118          0                 118   
7                        53         24                  52   
8                        15         15                   1   
9                        23          0                  23   
10                      178        174                 152   
11                      119         50                 119   
12                      246        238                 216   

         Primeiro alerta (s)                                 
modelo           Ensemble OU IF com PCA Regressão Logística  
Episódio                                                     
1                         45         46                  45  
2                         24         24                  73  
3                         24         24                  51  
4                          0         26                   0  
5                         14         21                  14  
6                          0          —                   0  
7                         26         27                  26  
8                         13         13                  40  
9                         94          —                  94  
10                         3          3                  16  
11                         0         35                   0  
12                        10         15                  10

### 6. Guardar os resultados desta comparação

Cada execução cria uma pasta nova em `runs/ensemble_...`. Em `previsoes_e_metricas/` ficam as tabelas e as previsões por linha. O `manifesto.json` registra a regra, os limiares e de onde vieram as previsões usadas.

In [6]:
agora = datetime.now().astimezone()
saida = runs / agora.strftime("ensemble_%Y%m%d_%H%M%S_%f")
tabelas = saida / "previsoes_e_metricas"
tabelas.mkdir(parents=True, exist_ok=False)
for nome, linhas, metricas, episodios in [
    ("validacao", linhas_val, metricas_val, episodios_val),
    ("teste_historico", linhas_teste, metricas_teste, episodios_teste),
]:
    metricas.to_csv(tabelas / f"{nome}_metricas.csv")
    episodios.to_csv(tabelas / f"{nome}_episodios.csv", index=False)
    linhas.to_csv(tabelas / f"{nome}_ensemble_previsoes.csv.gz", index=False)

manifesto = {
    "dataset": "HAI 21.03",
    "regra": "OU entre os alertas binários da Regressão Logística e do IF com PCA",
    "limiares": {"regressao_logistica": limiar_lr, "if_com_pca": limiar_if},
    "origem_limiares": "test4.csv.gz",
    "previsoes_logistica": str(pasta_lr.relative_to(raiz)),
    "previsoes_if_com_pca": str(pasta_if.relative_to(raiz)),
    "avaliacao": {"validacao": "test4.csv.gz", "teste_historico": "test5.csv.gz"},
    "executado_em": agora.isoformat(timespec="seconds"),
}
(saida / "manifesto.json").write_text(json.dumps(manifesto, indent=2, ensure_ascii=False), encoding="utf-8")
print("Resultados salvos em:", saida.relative_to(raiz))

Resultados salvos em: runs/ensemble_20261112_215609_211220
